In [9]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from dataset import FaceDataset,get_transforms
from model import get_efficientnet,get_xception
import os
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR="/kaggle/input/fake-face-detection/faces_split"
BATCH_SIZE=32
transform=get_transforms()
test_ds=FaceDataset(os.path.join(DATA_DIR,"test"),transform)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE)
def evaluate(model,weights_path,name):
    model.load_state_dict(torch.load(weights_path,map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    y_true, y_pred=[],[]
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs=imgs.to(DEVICE)
            outputs=model(imgs).squeeze()
            preds=(torch.sigmoid(outputs) > 0.5).int().cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(labels.numpy())
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred)
    }
eff_metrics=evaluate(get_efficientnet(),"/kaggle/input/facefhudn/efficientnet.pth","EfficientNet")
xcp_metrics=evaluate(get_xception(),"/kaggle/working/models/xception.pth","Xception")
print("\n📊 FINAL COMPARISON")
print(f"{'Model':<15}{'Acc':<10}{'Prec':<10}{'Recall':<10}{'F1':<10}")
print(f"{'EfficientNet':<15}{eff_metrics['Accuracy']:<10.4f}{eff_metrics['Precision']:<10.4f}{eff_metrics['Recall']:<10.4f}{eff_metrics['F1']:<10.4f}")
print(f"{'XceptionNet':<15}{xcp_metrics['Accuracy']:<10.4f}{xcp_metrics['Precision']:<10.4f}{xcp_metrics['Recall']:<10.4f}{xcp_metrics['F1']:<10.4f}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(



📊 FINAL COMPARISON
Model          Acc       Prec      Recall    F1        
EfficientNet   0.9414    0.9594    0.9209    0.9398    
XceptionNet    0.8160    0.7928    0.8518    0.8212    
